# 🎙️ Voice Studio

Clone a voice from a 10–30 second sample and make it speak **Hindi** or **English**.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thevishalvadher/voice-studio/blob/main/notebook.ipynb)

**How to use**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Run **1. Setup** (▶ button). First time: ~5 minutes. If the runtime restarts by itself ("session crashed"), that is normal: just run Setup again.
3. Run **2. Add Voice** to save a voice (only needed once per voice – it is kept in Google Drive).
4. Run **3. Generate** to make audio.

Everything is saved in Google Drive → `MyDrive/VoiceStudio/`.

⚖️ Only clone voices of people who gave you **signed consent**. Every output carries Resemble AI's invisible PerTh watermark (kept ON on purpose).

In [ ]:
#@title 1. Setup (run me first)
# First run: ~5 minutes (installs packages, downloads the model).
# Later runs in the same session: a few seconds.

REPO_URL = "https://github.com/thevishalvadher/voice-studio.git"
REPO_DIR = "/content/voice-studio"

import importlib.metadata, importlib.util, os, sys, time

# Step 1: get the latest code from GitHub
if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

# Step 2: install packages (only the first time in each runtime)
if importlib.util.find_spec("chatterbox") is None:
    print("⏳ Installing packages (about 3 minutes)…")
    !apt-get -qq install -y ffmpeg > /dev/null
    !pip install -q -r {REPO_DIR}/requirements.txt
    # Colab had already loaded older versions of some packages (like numpy).
    # If so, the runtime must restart once to use the new versions.
    stale = [n for n in ("numpy", "torch")
             if n in sys.modules and sys.modules[n].__version__ != importlib.metadata.version(n)]
    if stale:
        print("🔄 Restarting the runtime once to finish the install. This is normal.")
        print("👉 When it says 'Connected' again, run this Setup cell one more time.")
        sys.stdout.flush()
        time.sleep(2)
        os.kill(os.getpid(), 9)

# Step 3: connect Google Drive (a pop-up asks for permission)
from google.colab import drive
drive.mount("/content/drive")

# Step 4: create the Drive folders and load the voice model (once)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
from ui import notebook_ui
notebook_ui.setup()

## 2. Add Voice
Give the voice a name, pick its language, upload a sample **or** record one with your mic (10–30 seconds), tick the consent box, then click **Save Voice**.

In [ ]:
#@title 2. Add Voice
import sys
if "/content/voice-studio" not in sys.path:
    print("⚠️ Please run '1. Setup' first.")
else:
    from ui import notebook_ui
    notebook_ui.show_add_voice()

## 3. Generate
Pick a voice, type your text and click **Generate**. Files are saved to Drive in `outputs/<date>/`.

In [ ]:
#@title 3. Generate
import sys
if "/content/voice-studio" not in sys.path:
    print("⚠️ Please run '1. Setup' first.")
else:
    from ui import notebook_ui
    notebook_ui.show_generate()

## Optional: Engine test (Milestone 2)
Speaks one Hindi and one English test line with the model's built-in demo voice.
To test with your own clip instead, put its Drive path in `DEMO_SAMPLE` (e.g. `/content/drive/MyDrive/test.wav`).

In [ ]:
#@title Engine test
DEMO_SAMPLE = ""  #@param {type:"string"}
import sys
if "/content/voice-studio" not in sys.path:
    print("⚠️ Please run '1. Setup' first.")
else:
    from ui import notebook_ui
    notebook_ui.run_engine_test(DEMO_SAMPLE)